# TIDES on real events: one end-to-end test against the paper

This notebook runs the paper's evaluation protocol on one real EDS sequence and
checks the code against what the paper claims. The paper is Thirgood, Ghosh and
Hadfield, *TIDES: Time-Derivative Event Simulation via Deformable
Reconstruction*, arXiv:2606.02058.

| Step | What we test | Paper |
|---|---|---|
| 1 | Load one real RGB + event sequence (EDS) | Sec. 4.2 |
| 2 | Camera poses (COLMAP) and a fitted 4DGS scene | Sec. 3.1, Sec. 4.2 |
| 3 | The fitted scene renders the real frames | Sec. 3.1, Eq. 1-5; Sec. 5 ("bounded by the fidelity of the 4DGS scene") |
| 4 | The analytic time derivative L_dot matches the renderer | Sec. 3.2, Eq. 6-15; Fig. 1 |
| 5 | One contrast threshold, calibrated once and shared by every variant | Sec. 3.3; Sec. 4.6 ("sensor parameters fixed") |
| 6 | Simulate TIDES 10x, TIDES adaptive, frame-differencing baselines and ablations | Sec. 3.3-3.5, Sec. 4.1, Table 4 |
| 7 | Fidelity and batching metrics against the real events | Sec. 4.3, Tables 1-2, Eq. 20 |
| 8 | Automatic checks of the paper's claims | Tables 1, 2, 4 |
| 9 | Event frames and timestamp structure | Fig. 1, Fig. 3 |

**How to read the outcome.** The paper's metric settings (binning, windows,
normalisation) are in a supplementary file that is not public. Absolute numbers
will therefore be on a different scale from Tables 1-2. The test is whether the
**ranking and direction of every claim** hold on real data (step 8).

**Runtime on one A100:** COLMAP takes a few minutes, the 4DGS fit 30-60 min (skipped
when already done), and each simulated variant a few minutes for `SIM_SECONDS = 1`.

Set `USE_SYNTHETIC = True` to dry-run the whole notebook in a few minutes with
no data. Real events are then replaced by a densely simulated stream, so the
results say nothing about the paper; this mode only checks that everything runs.

In [ ]:
# ---- settings: edit these ------------------------------------------------------
USE_SYNTHETIC = False          # True: no data needed, plumbing check only
DATA_ROOT = "/u/marmary/airlab_v2e/evtween/data"   # Evtween-converted datasets
SEQ_NAME = "00_peanuts_dark"   # EDS sequence shown in Fig. 3 of the paper
NAME = "eds_00"                # name for the scene / model / outputs
MAX_FRAMES = 150               # frames used to fit the scene
STRIDE = 1                     # use every STRIDE-th frame; 2 or 3 if COLMAP registers too few frames
SIM_SECONDS = 1.0              # length of the simulated window (inside the fitted span)
DEVICE = "cuda"
# ---------------------------------------------------------------------------------

import glob, os, subprocess, sys, time, json
import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT)
from tides import TIDES, TIDESConfig
from tides.eval.metrics import Events, MetricConfig, all_metrics, per_pixel_isi
from tides.io import READERS, to_event_frame, save_events, load_events
from tides.render.projection import project_gaussians
from tides.render.rasterizer import rasterize_tangent

SCENE_DIR = f"{ROOT}/scenes/{NAME}"
MODEL_DIR = f"{ROOT}/outputs/4dgs/{NAME}"
OUT = f"{ROOT}/outputs/notebook/{NAME}" + ("_synthetic" if USE_SYNTHETIC else "")
os.makedirs(OUT, exist_ok=True)
if DEVICE == "cuda" and not torch.cuda.is_available():
    print("CUDA not available, falling back to CPU (slow)")
    DEVICE = "cpu"
print("TIDES repo:", ROOT)
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")

def run(cmd):
    # Run with this kernel's Python (Jupyter's PATH may point at another install).
    env = dict(os.environ, PYTHON=sys.executable)
    env["PATH"] = os.path.dirname(sys.executable) + os.pathsep + env.get("PATH", "")
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, cwd=ROOT, env=env)

if not USE_SYNTHETIC:
    # the official 4DGS code and its CUDA rasterizer are needed to fit the scene
    try:
        assert os.path.isdir(f"{ROOT}/third_party/4DGaussians")
        import diff_gaussian_rasterization, simple_knn  # noqa: F401
        print("4DGaussians: installed")
    except (AssertionError, ImportError) as e:
        raise SystemExit("4DGaussians is not installed. In a terminal on a GPU node, from the repo root:\n"
                         "  module load cuda/<version matching torch.version.cuda>\n"
                         "  bash scripts/install_4dgs.sh\n"
                         f"then restart this kernel. ({e!r})")

## 1. Real RGB + event sequence (Sec. 4.2)

The paper evaluates on four paired RGB-event datasets. EDS is one of them: it
captures ego-motion with a beam-splitter RGB + event rig (ref. [12]). We use the
Evtween conversion, which warps the RGB frames into the event camera's pixel
grid, so rendered frames and real events share one pixel grid.

In [ ]:
if USE_SYNTHETIC:
    SEQ = None
    print("synthetic mode: no dataset")
else:
    hits = sorted(glob.glob(f"{DATA_ROOT}/**/{SEQ_NAME}/frame_ts.npy", recursive=True))
    hits = [h for h in hits if "/eds/" in h.lower()] or hits
    assert hits, f"no {SEQ_NAME}/frame_ts.npy under {DATA_ROOT}; set SEQ by hand"
    SEQ = os.path.dirname(hits[0])
    from PIL import Image
    frames = sorted(glob.glob(f"{SEQ}/frames/*.png"))
    frame_ts = np.load(f"{SEQ}/frame_ts.npy")
    gt_all = READERS["evtween"](SEQ)  # still in the dataset's pixel grid; remapped in step 2
    W, H = Image.open(frames[0]).size
    print(f"sequence      {SEQ}")
    print(f"frames        {len(frames)}  {W}x{H}  fps {1/np.median(np.diff(frame_ts)):.1f}  span {frame_ts[-1]-frame_ts[0]:.2f}s")
    print(f"real events   {len(gt_all['t']):,}  ({len(gt_all['t'])/(gt_all['t'][-1]-gt_all['t'][0])/1e6:.2f} Mev/s)")
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].imshow(Image.open(frames[0])); ax[0].set_title("RGB frame 0")
    t0 = frame_ts[0]
    ax[1].imshow(to_event_frame(gt_all, W, H, t0, t0 + 0.01)); ax[1].set_title("real events, first 10 ms")
    for a in ax: a.axis("off")
    plt.show()

## 2. Camera poses and the dynamic Gaussian scene (Sec. 3.1, Sec. 4.2)

> "When 6-DoF trajectories are provided, we use the official poses; otherwise,
> we estimate camera motion and sparse geometry from RGB using COLMAP for static
> scenes [...] For each sequence we fit a dynamic Gaussian splatting scene."
> (Sec. 4.2)

The Evtween frames sit in the event camera's frame, not the frame of EDS's
ground-truth poses, so we use COLMAP. EDS scenes are static apart from the
camera, which is exactly the COLMAP case in the paper. The scene is then fit
with the official 4DGS code (ref. [37]), the same substrate the paper uses.
Both steps are skipped if their outputs already exist. You can also run them
in a terminal or an sbatch job, using the same two commands.

In [ ]:
from tides.sequence import Sequence
if USE_SYNTHETIC:
    from tides.scene.synthetic import occlusion_scene, static_trajectory
    W, H = SYN_SIZE if "SYN_SIZE" in globals() else (128, 96)
    scene = occlusion_scene(speed=2.0).float()
    traj = static_trajectory(W, H, t1=0.6, n=19, pan=0.2, dtype=torch.float32)
    t_fit0, t_fit1 = traj.t_start, traj.t_end
else:
    sj = f"{SCENE_DIR}/tides_sequence.json"
    if os.path.exists(sj) and "normalize" not in Sequence.load(sj).meta:
        print("scene was built by an older version of prepare_scene.py: rebuilding it and the 4DGS fit")
        import shutil
        shutil.rmtree(SCENE_DIR, ignore_errors=True); shutil.rmtree(MODEL_DIR, ignore_errors=True)
    if not os.path.exists(sj):
        run([sys.executable, "scripts/prepare_scene.py", "--preset", "evtween", "--seq_dir", SEQ,
             "--out", SCENE_DIR, "--max_frames", str(MAX_FRAMES), "--stride", str(STRIDE), "--run_colmap"])
    seq = Sequence.load(sj)
    print(f"COLMAP registered {len(seq.frames)} frames, {seq.t1 - seq.t0:.2f}s, intrinsics {np.round(seq.intrinsics, 1)}")
    # Frames were resampled to a centred, undistorted pinhole camera (what 4DGS assumes);
    # move the real events to the same pixel grid.
    gt_all = seq.remap_events(gt_all)
    print(f"real events on the {seq.size[0]}x{seq.size[1]} pinhole grid: {len(gt_all['t']):,}")
    if not glob.glob(f"{MODEL_DIR}/point_cloud/iteration_*"):
        run(["bash", "scripts/fit_4dgs.sh", SCENE_DIR, NAME])
    from tides.scene.deformation import Upstream4DGS
    scene = Upstream4DGS.from_model_dir(MODEL_DIR, -1, f"{ROOT}/third_party/4DGaussians")
    scene.set_time_mapping(seq.t0, seq.t1)  # 4DGS time is normalised to [0,1]
    traj = seq.trajectory("rgb")
    W, H = traj.intr[4], traj.intr[5]
    t_fit0, t_fit1 = seq.t0, seq.t1
    print(f"4DGS scene: {scene.g.get_xyz.shape[0]:,} Gaussians")
scene = scene.to(DEVICE)
BG = tuple(getattr(scene, "background", (0.0, 0.0, 0.0)))  # 4DGS trains over white by default

@torch.no_grad()
def render(t, **kw):
    g = scene.state(t)
    cam = traj(t, dtype=g.means.dtype, device=DEVICE)
    kw.setdefault("bg", torch.tensor(BG, dtype=g.means.dtype, device=DEVICE))
    return rasterize_tangent(project_gaussians(g, cam), cam.width, cam.height, **kw)

## 3. Check A: the scene reproduces the real frames (Sec. 3.1, Eq. 1-5)

TIDES renders with EWA splatting and front-to-back compositing (Eq. 1-4). We
render the fitted scene with **our** time-tangent rasterizer, not 4DGS's own
renderer, and compare with the real frames. This checks two things: the 4DGS
fit, and that our rasterizer reproduces the 4DGS image formation.

> "The quality of events simulated through TIDES is ultimately bounded by the
> fidelity of the underlying 4DGS scene reconstruction." (Sec. 5)

**Pass:** mean PSNR above about 25 dB. Below about 20 dB the events cannot be
expected to match, so fix the scene fit before reading the later steps.

In [ ]:
def psnr(a, b):
    return float(10 * np.log10(1.0 / max(np.mean((a - b) ** 2), 1e-12)))

if USE_SYNTHETIC:
    print("synthetic mode: the scene is exact, PSNR check skipped")
    PSNR_OK = True
else:
    from PIL import Image
    idx = np.linspace(0, len(seq.frames) - 1, 6).round().astype(int)
    vals = []
    fig, ax = plt.subplots(2, len(idx), figsize=(3 * len(idx), 5))
    for j, i in enumerate(idx):
        r = render(float(seq.timestamps[i]))
        pred = r.rgb.clamp(0, 1).permute(1, 2, 0).cpu().numpy()
        real = np.asarray(Image.open(seq.frames[i]).convert("RGB"), dtype=np.float32) / 255
        vals.append(psnr(pred, real))
        ax[0, j].imshow(real); ax[0, j].set_title(f"real #{i}")
        ax[1, j].imshow(pred); ax[1, j].set_title(f"TIDES render {vals[-1]:.1f} dB")
    for a in ax.ravel(): a.axis("off")
    plt.tight_layout(); plt.show()
    PSNR_OK = np.mean(vals) > 25
    print(f"mean PSNR {np.mean(vals):.2f} dB  ->  {'PASS' if PSNR_OK else 'CHECK THE SCENE FIT'}")

## 4. Check B: analytic time derivative (Sec. 3.2, Eq. 6-15; Fig. 1)

The paper's central idea is that L_dot is computed **analytically**: one
forward-mode JVP through the deformation network gives Theta_dot (Eq. 6-8), and
it is pushed through the projection and the front-to-back compositing (Eq. 9-14)
to give L_dot = Y_dot / (Y + eps) (Eq. 15).

The check runs **in float64**, stage by stage, on the fitted scene: raw
deformation outputs (Eq. 6), activations (Eq. 7-8), camera, projection, and the
composited rgb, alpha and L (Eq. 9-15). Each analytic derivative is compared
with a central finite difference with a step of 1e-5 frame intervals. Elements
where the function itself is not differentiable inside that tiny step (a ReLU
kink, a HexPlane time knot, a splat crossing the 1/255 cut-off) are found from
the left and right one-sided differences and left out; they are counted.

**Pass:** in every stage, at least 99.9% of the remaining elements agree to a
relative 1e-4. A 1% error in any stage fails (tests/test_derivative_check.py).
The verdict is saved to `derivative_check.json` next to the outputs. The same
check runs from a terminal with `python scripts/check_derivative.py`.

Why not float32: there (f(t+h) - f(t-h)) / 2h loses its digits to round-off
unless h is large, and a large h crosses the discontinuities above, so the
finite difference itself is unreliable. That is why the earlier float32 version
of this check could only say "inconclusive".

In [ ]:
from tides.eval.derivative_check import run_check
dtype0 = next(iter(list(scene.parameters()) + list(scene.buffers()))).dtype
scene.to(torch.float64)
try:
    CHK = run_check(TIDES(scene, traj, TIDESConfig(device=DEVICE, background=BG)))
finally:
    scene.to(dtype0)
os.makedirs(OUT, exist_ok=True)
with open(f"{OUT}/derivative_check.json", "w") as f:
    json.dump(CHK, f, indent=2)
DERIV_OK = CHK["verdict"] == "PASS"

Fig. 1 (left half): L, the analytic L_dot, transmittance and its derivative,
at the middle of the clip.

In [ ]:
t_mid = 0.5 * (t_fit0 + t_fit1)
render(t_mid)  # warm-up
tic = time.time(); r = render(t_mid); torch.cuda.synchronize() if DEVICE == "cuda" else None; t_one = time.time() - tic
print(f"one tangent render (L and L_dot together) {t_one*1000:.0f} ms")
an = r.L_dot
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
lim = float(an.abs().quantile(0.99)) + 1e-9
ax[0].imshow(r.L.cpu(), cmap="gray"); ax[0].set_title("L = log(Y + eps)  (Eq. 5)")
ax[1].imshow(an.cpu(), cmap="RdBu_r", vmin=-lim, vmax=lim); ax[1].set_title("L_dot analytic (Eq. 15)")
ax[2].imshow((1 - r.alpha).cpu(), cmap="gray"); ax[2].set_title("transmittance T_K")
lim2 = float(r.alpha_dot.abs().quantile(0.99)) + 1e-9
ax[3].imshow((-r.alpha_dot).cpu(), cmap="RdBu_r", vmin=-lim2, vmax=lim2); ax[3].set_title("d T_K / dt = -alpha_dot (Eq. 13)")
for x in ax: x.axis("off")
plt.tight_layout(); plt.show()

## 5. One shared contrast threshold (Sec. 3.3, Sec. 4.6)

The paper keeps "the rendering source, sensor parameters, and evaluation
protocol fixed" across variants (Sec. 4.6), but it does not publish the
threshold C. We calibrate C **once**: we pick the C at which TIDES (10x)
produces as many events as the real camera over a short window. Every variant
then uses that same C. This is the fair version of "same sensor parameters".

In [ ]:
sim0 = t_fit0 + 0.1 * (t_fit1 - t_fit0)
sim1 = min(sim0 + SIM_SECONDS, t_fit1)
if USE_SYNTHETIC:
    ref_cfg = TIDESConfig(device=DEVICE, queries_per_frame=SYN_REF_Q if "SYN_REF_Q" in globals() else 60, adaptive=False)
    ref_cfg.comparator.C_pos = ref_cfg.comparator.C_neg = 0.15
    ref_cfg.comparator.n_max = 16
    gt_all = TIDES(scene, traj, ref_cfg).simulate(sim0, sim1, verbose=False)
    print(f"synthetic 'real' events: {len(gt_all['t']):,} (dense TIDES, 60 queries/frame)")
GT = Events.from_dict(gt_all).window(sim0, sim1)
print(f"evaluation window [{sim0:.3f}, {sim1:.3f}] s, real events {len(GT):,}")

import yaml
def cfg_from(files=(), **over):
    d = {}
    for p in files:
        for k, v in (yaml.safe_load(open(f"{ROOT}/{p}")) or {}).items():
            d[k] = {**d.get(k, {}), **v} if isinstance(v, dict) else v
    for k, v in over.items():
        if "." in k:
            a, b = k.split(".")
            d.setdefault(a, {})[b] = v
        else:
            d[k] = v
    d["device"] = DEVICE
    d.setdefault("background", list(BG))
    return TIDESConfig.from_dict(d)

cal1 = min(sim0 + 0.2, sim1)
n_real = len(GT.window(sim0, cal1))
counts = {}
for C in (0.03, 0.04, 0.05, 0.07, 0.1, 0.15, 0.2, 0.3, 0.5):
    cfg = cfg_from(["configs/tides_10x.yaml"], **{"comparator.C_pos": C, "comparator.C_neg": C, "arbiter.enabled": False})
    counts[C] = len(TIDES(scene, traj, cfg).simulate(sim0, cal1, verbose=False)["t"])
    print(f"  C = {C:.2f}: {counts[C]:,} events (real: {n_real:,})")
Cs = np.array(list(counts)); ns = np.array(list(counts.values()), dtype=float)
C_STAR = float(np.exp(np.interp(np.log(n_real), np.log(ns[::-1] + 1), np.log(Cs[::-1]))))
if n_real > max(counts.values()):
    print("[warn] even the lowest C gives fewer events than the real camera; C is clamped to the grid edge")
print(f"calibrated contrast threshold C = {C_STAR:.3f}  (used by every variant below)")

## 6. Simulate every variant from the same scene (Sec. 3.3-3.5, Sec. 4.1, Table 4)

> "Frame-driven baselines are given renders at regular intervals [...] This
> isolates differences in time-stamping, thresholding, readout modeling, and
> occlusion handling from interpolation artifacts in the inputs." (Sec. 4.2)

| Variant | What it is | Paper |
|---|---|---|
| `FD 1x` | slope by differencing two renders one frame apart, linear crossing; all other settings as TIDES (10x). A plain frame-based simulator | Sec. 2.1, Sec. 4.2 |
| `FD 10x (ESIM-style)` | the same with the 10x render budget | Table 1 "ESIM"; Table 4 "Finite-diff slope" |
| `TIDES (10x)` | analytic L_dot, quadratic local model, multi-crossing, masking, readout arbiter | Table 1 "TIDES (10x)" |
| `TIDES (adaptive)` | plus risk-guided adaptive sub-steps (Eq. 18) | Table 1 "TIDES (adaptive)" |
| `no multi-crossing` | adaptive, at most one event per pixel per step | Table 4 |
| `no readout` | adaptive, ideal timestamps (no Eq. 19 arbiter) | Table 4 |

Results are cached in `OUT`. Delete the `.npz` files to re-run. Each `.npz` gets a
`.json` next to it (time window, C, commit, derivative verdict) that Evtween's
`compare_final.ipynb` reads to import the stream.

In [ ]:
C_set = {"comparator.C_pos": C_STAR, "comparator.C_neg": C_STAR}
# The FD baselines share every sensor setting with TIDES (10x) and change only the slope:
# exactly the Table 4 ablation "Finite-diff slope (L_dot by FD)".
FD = ["configs/tides_10x.yaml", "configs/ablations/fd_slope.yaml"]
VARIANTS = {
    "FD 1x":               cfg_from(FD, queries_per_frame=1, **C_set),
    "FD 10x (ESIM-style)": cfg_from(FD, **C_set),
    "TIDES (10x)":         cfg_from(["configs/tides_10x.yaml"], **C_set),
    "TIDES (adaptive)":    cfg_from(["configs/tides_adaptive.yaml"], **C_set),
    "no multi-crossing":   cfg_from(["configs/tides_adaptive.yaml", "configs/ablations/no_multicrossing.yaml"], **C_set),
    "no readout":          cfg_from(["configs/tides_adaptive.yaml", "configs/ablations/no_readout.yaml"], **C_set),
}
COMMIT = subprocess.run(["git", "-C", ROOT, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() or "unknown"

def write_sidecar(path, name, cfg):
    # Everything a benchmark needs to import this stream without guessing (read by Evtween's compare_final.ipynb).
    side = path[:-4] + ".json"
    if os.path.exists(side):
        return
    info = dict(variant=name, coverage_s=[float(sim0), float(sim1)], timestamp_unit="s", timestamp_origin_s=0.0,
                coordinate_space="pinhole", resolution_wh=[int(W), int(H)],
                scene_json=None if USE_SYNTHETIC else f"{SCENE_DIR}/tides_sequence.json",
                synthetic=bool(USE_SYNTHETIC), contrast_threshold=float(C_STAR),
                gt_count_calibrated=True, calibration="C chosen so TIDES (10x) matches the real event count "
                                                       f"in [{sim0:.3f}, {cal1:.3f}] s (step 5)",
                derivative_check=CHK["verdict"], derivative_check_file=f"{OUT}/derivative_check.json",
                commit=COMMIT, config=cfg.to_dict())
    with open(side, "w") as f:
        json.dump(info, f, indent=2, default=str)

SIM = {}
for name, cfg in VARIANTS.items():
    path = f"{OUT}/{name.replace(' ', '_').replace('(', '').replace(')', '')}.npz"
    if os.path.exists(path):
        SIM[name] = load_events(path); write_sidecar(path, name, cfg)
        print(f"{name:22s} cached  {len(SIM[name]['t']):,} events"); continue
    out = TIDES(scene, traj, cfg).simulate(sim0, sim1, verbose=False)
    save_events(path, out); SIM[name] = out
    write_sidecar(path, name, cfg)
    s = out["stats"]
    print(f"{name:22s} {len(out['t']):>10,} events  {s['wall_time_s']:7.1f}s  renders {s['full_renders']} full + {s['tile_renders']} tiles")

## 7. Metrics against the real events (Sec. 4.3, Tables 1-2, Eq. 20)

* **IG-NLL** (lower is better): NLL of the real per-pixel inter-spike intervals
  under an inverse-Gaussian fitted to the simulator's ISIs ("DVS-Voltmeter-style").
* **Chamfer** (lower is better): bidirectional nearest neighbour in the
  motion-scaled embedding (x, y, v t), polarity matched, voxel density control.
* **Same-ts / ISI spike / Fano** (lower is better): batching diagnostics; Fano
  error is Eq. 20.

Our settings are in `configs/metrics.yaml` and `docs/assumptions.md`, item 11.

In [ ]:
import pandas as pd
mcfg = MetricConfig(**yaml.safe_load(open(f"{ROOT}/configs/metrics.yaml")))
rows = {}
for name, ev in SIM.items():
    P = Events.from_dict(ev).window(sim0, sim1)
    rows[name] = all_metrics(GT, P, W, H, mcfg)
from tides.eval.metrics import same_ts_fraction, isi_spike_ratio
real_diag = {"Same-ts": same_ts_fraction(GT, mcfg), "ISI-spike": isi_spike_ratio(GT, W, mcfg)}
RES = pd.DataFrame(rows).T[["IG-NLL", "Chamfer", "Same-ts", "Fano", "ISI-spike", "n_sim", "n_real"]]
RES.to_csv(f"{OUT}/metrics.csv")
print("real camera itself: Same-ts %.4f  ISI-spike %.4f" % (real_diag["Same-ts"], real_diag["ISI-spike"]))
RES.style.format({"IG-NLL": "{:.5f}", "Chamfer": "{:.5f}", "Same-ts": "{:.4f}", "Fano": "{:.4f}", "ISI-spike": "{:.4f}",
                  "n_sim": "{:,.0f}", "n_real": "{:,.0f}"}).highlight_min(subset=["IG-NLL", "Chamfer", "Fano"], color="#ffd6a5")

## 8. Does the code reproduce the paper's claims? (Tables 1, 2, 4)

Paper values for EDS, for reference:

| Simulator | IG-NLL | Chamfer | Same-ts | Fano | ISI spike |
|---|---|---|---|---|---|
| ESIM | 0.004748 | 0.046932 | 0.281985 | 0.506379 | 0.096938 |
| TIDES (10x) | 0.003840 | 0.043245 | 0.040882 | 0.228440 | 0.052497 |
| TIDES (adaptive) | 0.003733 | 0.042622 | 0.025234 | 0.338062 | 0.055599 |
| Table 4, finite-diff slope | 0.00521 | 0.04801 | | | |
| Table 4, no multi-crossing | 0.00439 | 0.04487 | | | |
| Table 4, no readout | 0.00388 | 0.04294 | | | |

Each check below tests one claim, by its direction rather than its absolute value.

In [ ]:
R = RES
def better(a, b, col):  # a strictly better (lower) than b
    return bool(R.loc[a, col] < R.loc[b, col])
CHECKS = [
    ("Scene renders the real frames (PSNR > 25 dB)", "Sec. 5", PSNR_OK),
    ("Analytic derivatives match float64 finite differences, every stage", "Sec. 3.2, Eq. 6-15", DERIV_OK),
    ("TIDES 10x beats ESIM-style FD 10x on IG-NLL", "Table 1 (ESIM vs TIDES 10x); Table 4 row 2", better("TIDES (10x)", "FD 10x (ESIM-style)", "IG-NLL")),
    ("TIDES 10x beats ESIM-style FD 10x on Chamfer", "Table 1; Table 4 row 2", better("TIDES (10x)", "FD 10x (ESIM-style)", "Chamfer")),
    ("TIDES 10x has less batching (Same-ts) than FD 10x", "Table 2", bool(R.loc["TIDES (10x)", "Same-ts"] <= R.loc["FD 10x (ESIM-style)", "Same-ts"])),
    ("TIDES 10x has lower Fano error than FD 10x", "Table 2", better("TIDES (10x)", "FD 10x (ESIM-style)", "Fano")),
    ("Adaptive is no worse than 10x on Chamfer", "Table 1 (adaptive row)", bool(R.loc["TIDES (adaptive)", "Chamfer"] <= R.loc["TIDES (10x)", "Chamfer"] * 1.01)),
    ("Removing multi-crossing hurts IG-NLL or Chamfer", "Table 4", better("TIDES (adaptive)", "no multi-crossing", "IG-NLL") or better("TIDES (adaptive)", "no multi-crossing", "Chamfer")),
    ("Removing the readout changes little (< 5% Chamfer)", "Table 4 ('smaller but consistent drop')",
     bool(abs(R.loc["no readout", "Chamfer"] - R.loc["TIDES (adaptive)", "Chamfer"]) <= 0.05 * R.loc["TIDES (adaptive)", "Chamfer"])),
    ("More renders help frame differencing (FD 10x beats FD 1x on Chamfer)", "Sec. 1 ('mitigate but not fully remove')", better("FD 10x (ESIM-style)", "FD 1x", "Chamfer")),
]
for claim, ref, ok in CHECKS:
    print(f"{'PASS' if ok else 'FAIL'}  {claim:70s} [{ref}]")
n_ok = sum(ok for _, _, ok in CHECKS)
print(f"\n{n_ok}/{len(CHECKS)} checks pass")
json.dump([dict(claim=c, paper=r, passed=bool(o)) for c, r, o in CHECKS], open(f"{OUT}/checks.json", "w"), indent=2)

## 9. What the events look like (Fig. 1, Fig. 3)

The top row has event frames over 10 ms (red = ON, blue = OFF), as in Fig. 3.
The bottom left shows the phase of each event timestamp within its render step.
A simulator that batches timestamps piles up at fixed phases; continuous-time
crossings spread evenly (Fig. 1, "timestamp batching"). The bottom right shows
the per-pixel inter-spike-interval distributions against the real camera
(the quantity behind IG-NLL).

In [ ]:
show = ["FD 10x (ESIM-style)", "TIDES (10x)", "TIDES (adaptive)"]
tm = 0.5 * (sim0 + sim1)
fig, ax = plt.subplots(2, 4, figsize=(18, 8))
ax[0, 0].imshow(to_event_frame(gt_all, W, H, tm, tm + 0.01)); ax[0, 0].set_title("real events")
for j, n in enumerate(show):
    ax[0, j + 1].imshow(to_event_frame(SIM[n], W, H, tm, tm + 0.01)); ax[0, j + 1].set_title(n)
for a in ax[0]: a.axis("off")

frame_dt = float(np.median(np.diff(traj.times.numpy())))
step = frame_dt / 10
for n in show:
    ph = ((SIM[n]["t"] - sim0) / step) % 1.0
    ax[1, 0].hist(ph, bins=100, histtype="step", density=True, label=n)
ax[1, 0].set_xlabel("timestamp phase within a render step"); ax[1, 0].set_ylabel("density"); ax[1, 0].legend(fontsize=8)
ax[1, 0].set_title("timestamp batching (flat = continuous)")

bins = np.logspace(-6, 0, 80)
ax[1, 1].hist(per_pixel_isi(GT, W), bins=bins, histtype="step", density=True, color="k", lw=2, label="real")
for n in show:
    ax[1, 1].hist(per_pixel_isi(Events.from_dict(SIM[n]).window(sim0, sim1), W), bins=bins, histtype="step", density=True, label=n)
ax[1, 1].set_xscale("log"); ax[1, 1].set_xlabel("per-pixel ISI (s)"); ax[1, 1].legend(fontsize=8); ax[1, 1].set_title("ISI distribution (IG-NLL)")

rate_t = np.arange(sim0, sim1, 0.005)
ax[1, 2].plot(rate_t[:-1], np.histogram(GT.t, rate_t)[0] / 0.005 / 1e6, "k", lw=2, label="real")
for n in show:
    ax[1, 2].plot(rate_t[:-1], np.histogram(SIM[n]["t"], rate_t)[0] / 0.005 / 1e6, label=n)
ax[1, 2].set_xlabel("t (s)"); ax[1, 2].set_ylabel("Mev/s"); ax[1, 2].legend(fontsize=8); ax[1, 2].set_title("event rate")

RES[["IG-NLL", "Chamfer", "Fano"]].rank().plot.barh(ax=ax[1, 3]); ax[1, 3].set_title("rank per metric (1 = best)")
plt.tight_layout(); plt.savefig(f"{OUT}/figure.png", dpi=120); plt.show()

## 10. Reading the result

* **Steps 3 and 4 pass, and most of step 8 passes:** the reimplementation
  reproduces the paper's claims on this sequence. To report it like Table 1,
  repeat with other EDS sequences (change `SEQ_NAME` and `NAME`) and average.
* **Step 3 fails:** the scene fit is the bottleneck (Sec. 5). Improve the 4DGS
  fit first, for example with more iterations or a better COLMAP model.
  Nothing downstream can be judged until it passes.
* **Step 4 fails:** send me the output of that cell. It points to a mismatch
  between our renderer and the 4DGS checkpoint.
* **Only the batching checks differ:** our frame-differencing baseline already
  interpolates crossings linearly, as ESIM does. The paper's ESIM run shows
  stronger batching, which probably comes from details of ESIM's implementation.
  The fair comparison against the real ESIM, V2E and DVS-Voltmeter is to run
  them from Evtween's `benchmark/event-generation` branch on the same window
  and score them here with `all_metrics`.

Everything this run produced is in `OUT`: events, `metrics.csv`, `checks.json`, and `figure.png`.